# Rohdatenextrakte von MPG.PuRe

Dieses Skript wird mit Datendownloads aus PuRe nach eigenem Inhaltszuschnitt mit generischen Namen versorgt. Diese werden in unterschiedlichen Schritte aufbereitet, um in verschiedenen Szenarien in eigenen Juypter Notebooks ausgwertet zu werden.   

Hinweise mit Handlungsbedarf (Bestimmte Feldanpassungen oder Auswahlkriterien für eigenes Institut) sind als Zwischenüberschriften gekennzeichnet. Grundsätzlich sind Hinweise zum Aufbau des Codes als Auskommentierungen im Code selbst zu finden. Die Idee ist, dass der Code ggf. mit institutsspezifischen Anpassungen grundsätzlich einfach durchläuft.

In [1]:
# import der benötigten Pakete, bei Bedarf über pip install <paketname> installieren
import pandas as pd
import numpy as np
import re
import pdb
from collections import defaultdict
import os

## Datenimport über Openrefine

**Da das JSON aus PuRe nicht direkt in Python einlesbar ist, geht mein Bearbeitungsweg über Openrefine!**   

Beschreibung hierzu, siehe (https://github.com/cogemol/Repository_Data/blob/main/Datenexport%20aus%20PuRe%20f%C3%BCr%20Skripte%20via%20Openrefine.pdf)

In [2]:
# Einlesen der Daten nach Export aus Openrefine und Ablage der Datei im Verzeichnis "Daten"
df_pure = pd.read_excel('Daten/download-json.xlsx')  # Dateiname ggf. anpassen


c:\Users\cgmol\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [3]:
# Durchzählen der IDs, um sicherzustellen, dass alle Datensätze eingelesen wurden
df_pure['data - objectId'].count() 

np.int64(447)

In [4]:
# Umbenennung von Spalten, um die Verständlichkeit zu erhöhen und die weitere Bearbeitung zu erleichtern 

df_pure = df_pure.rename(columns={
    'data - metadata - title':'1001_Titel',
    'data - metadata - genre':'1002_Genre',
    'data - metadata - degree' : '1021_Degree',
    'data - metadata - reviewMethod':'1108_Review',
    'data - metadata - datePublishedInPrint':'1102_Erscheinungsdatum',
    'data - metadata - datePublishedOnline':'1103_OnlineDatum',
    'data - metadata - creators - _ - role':'0013_Creator-Rolle',
    'data - metadata - creators - _ - type':'Creator-Typ', 
    'data - metadata - creators - _ - person - givenName':'2_Vorname',  
    'data - metadata - creators - _ - person - familyName':'1_Nachname',
    'data - metadata - creators - _ - person - identifier - type' :'4_Cone',
    'data - metadata - creators - _ - person - organizations - _ - name':'5_Affiliation',
    'data - metadata - sources - _ - title':'1101_Quellentitel',
    'data - metadata - sources - _ - genre':'1100_Quellen-Genre',
    'data - metadata - sources - _ - startPage':'1106_Anfangsseite',
    'data - metadata - sources - _ - endPage':'1107_Endseite',
    'data - metadata - sources - _ - issue':'1105_Heft', 
    'data - metadata - sources - _ - volume':'1104_Band',
    'data - metadata - sources - _ - sequenceNumber' : '1108_Sequenznummer',
    'data - metadata - sources - _ - identifiers - _ - id':'1109_Identifier',
    'data - metadata - sources - _ - creators - _ - role':'1110_Quellcreator-Rolle',
    'data - metadata - sources - _ - creators - _ - type':'Quellcreator-Type',
    'data - metadata - sources - _ - creators - _ - person - givenName':'2_Quellcreator-Vorname',
    'data - metadata - sources - _ - creators - _ - person - familyName':'1_Quellcreator-Nachname',
    'data - metadata - sources - _ - creators - _ - person - orcid':'Quellcreator-ORCID',
    'data - metadata - sources - _ - creators - _ - person - organizations - _ - name' :'Quellcreator-Affiliation',
    'data - metadata - sources - _ - creators - _ - person - identifier - type':'Quellcreator-Cone',
    'data - metadata - sources - _ - alternativeTitles - _ - value':'Quellentitel-Alternativ',
    'data - metadata - sources - _ - publishingInfo - publisher' : 'Quellenverlag',
    'data - metadata - sources - _ - publishingInfo - place' : 'Quellenpublikationsort',
    'data - metadata - identifiers - _ - type':'2201_Identifier',
    'data - metadata - identifiers - _ - id':'2202_ID',
    'data - metadata - languages - languages':'2200_Sprache',
    'data - metadata - publishingInfo - place' : 'Verlagsort',
    'data - metadata - publishingInfo - publisher' : 'Verlag',
    'data - metadata - projectInfo - _ - title':'Projekt',
    'data - metadata - projectInfo - _ - grantIdentifier - type' : 'Projekt-Identifier',
    'data - metadata - projectInfo - _ - grantIdentifier - id' : 'Projekt-ID',
    'data - metadata - projectInfo - _ - fundingInfo - fundingOrganization - title' : 'Projekt-Förderer',
    'data - localTags - localTags':'0_LocalTags',
    'data - files - _ - content':'4_File-content',
    'data - files - _ - metadata - title':'2_File-title',
    'data - files - _ - visibility':'5_File-visibility',
    'data - files - _ - storage':'7_File-Storage',
    'data - files - _ - metadata - contentCategory':'3_File-Category',
    'data - files - _ - metadata - description':'1_File-Description',
    'data - files - _ - metadata - oaStatus':'6_File-OA',
    'data - files - _ - metadata - license':'8_File-Licence',
    'data - files - _ - metadata - embargoUntil':'9_File-Embargo',
    'data - metadata - abstracts - _ - value' : '90991_Abstract',
    'data - metadata - freeKeywords' : '90992_Stichworte',
    'data - objectId':'90999_ItemID',}) 
#df_pure

In [5]:
# Bereinigungs- und Auffüllarbeiten

# Auffüllen der ID-Spalte, damit auch hier die IDs zu finden sind und später korrekte Zuordnung möglich ist
df_pure['90999_ItemID'] = df_pure['90999_ItemID'].ffill() 

# Auffüllen der Genre-Spalte
df_pure['1002_Genre'] = df_pure['1002_Genre'].ffill()   

# Hinzufügen eines Zeilenzählers für die einzelnen IDs 
df_pure['Item_seq'] = df_pure.groupby('90999_ItemID').cumcount() + 1 

# Einfügen einer Spalte für kompletten Namen
df_pure['Name'] = df_pure['2_Vorname'].str.cat(df_pure['1_Nachname'], sep=' ')

In [6]:
# Bereinigung der Jahreszahlen und immer Auswahl des ersten Datums

# Schritt 1: Daten reduzieren auf Jahr
df_pure['1102_Erscheinungsdatum'] = df_pure['1102_Erscheinungsdatum'].str.extract(r'(\d{4})')
df_pure['1103_OnlineDatum'] = df_pure['1103_OnlineDatum'].str.extract(r'(\d{4})')

# Schritt 2: Daten umwandeln in Zahlen
df_pure['1102_Erscheinungsdatum'] = pd.to_numeric(df_pure['1102_Erscheinungsdatum'], errors='coerce').astype('Int64')
df_pure['1103_OnlineDatum'] = pd.to_numeric(df_pure['1103_OnlineDatum'], errors='coerce').astype('Int64')

# Schritt 3: jeweils die kleinste Zahl nehmen und in neue Spalte schreiben
df_pure['kleinster_wert'] = df_pure[['1102_Erscheinungsdatum', '1103_OnlineDatum']].min(axis=1)

# Schritt 4: Extrahiere das Jahr aus dem kleinsten Wert (als String)
df_pure['1103_Ersterscheinungsdatum'] = df_pure['kleinster_wert'].astype(str).str.extract(r'(\d{4})')

In [7]:
df_pure['1103_Ersterscheinungsdatum'].value_counts()

1103_Ersterscheinungsdatum
2024    136
2023    133
2025    130
2022     31
2026     10
2021      5
2017      1
2020      1
Name: count, dtype: int64

In [8]:
# Kopie der Rohdaten zur Weiterverarbeitung in den anderen Notebooks (da auch Abstracts enthalten sind, kein Export als CSV, da sonst Fehler entstehen können)
df_pure.to_excel('Daten/df_pure.xlsx', index=False)

# Umbenennung des df, um Unterscheidung zu erleichtern
merged_df = df_pure 

In [9]:
# Code, der Spalten einer ID ausliest, und in neue Spalten in erste Zeile umschreibt, hier: Nachname
# Geändert werden muss immer der Wert in "column_name" zur entsprechenden Spalte und Values

column_name = '1_Nachname'

df_pivoted_add_column = merged_df.pivot_table(
    index='90999_ItemID',           # Use itemID as the index for the new DataFrame
    columns='Item_seq',    # Use vorname_seq to create new columns
    values=column_name,         # The values for the new columns 
    aggfunc='first'           # In case of multiple values for a seq (unlikely here)
)

def consolidate_values(row):
    # Get non-NaN values from the row, preserving order
    values = [value for value in row if pd.notna(value)]
    return values

# Apply the function to each row to get a list of consolidated values
consolidated_lists = df_pivoted_add_column.apply(consolidate_values, axis=1)

# Create new columns based on the consolidated lists
max_values = consolidated_lists.apply(len).max() # Find the maximum number of values in any row

new_added_cols = {}
for i in range(max_values):
    new_col_name = f'{i + 1}{column_name}'
    new_added_cols[new_col_name] = consolidated_lists.apply(lambda x: x[i] if i < len(x) else None)

# Create a new DataFrame with the 'ID' and the new 'Vorname_' columns
df_pivoted_added_column = pd.DataFrame(new_added_cols)

merged_df = pd.merge(merged_df, df_pivoted_added_column, on='90999_ItemID', how='left', suffixes=('_original', '_pivoted'))

In [10]:
# Code, der Spalten einer ID ausliest, und in neue Spalten in erste Zeile umschreibt, hier: Vorname  
# Geändert werden muss immer der Wert in "column_name" zur entsprechenden Spalte und Values

column_name = '2_Vorname'

df_pivoted_add_column = merged_df.pivot_table(
    index='90999_ItemID',           # Use itemID as the index for the new DataFrame
    columns='Item_seq',    # Use vorname_seq to create new columns
    values=column_name,         # The values for the new columns 
    aggfunc='first'           # In case of multiple values for a seq (unlikely here)
)

def consolidate_values(row):
    # Get non-NaN values from the row, preserving order
    values = [value for value in row if pd.notna(value)]
    return values

# Apply the function to each row to get a list of consolidated values
consolidated_lists = df_pivoted_add_column.apply(consolidate_values, axis=1)

# Create new columns based on the consolidated lists
max_values = consolidated_lists.apply(len).max() # Find the maximum number of values in any row

new_added_cols = {}
for i in range(max_values):
    new_col_name = f'{i + 1}{column_name}'
    new_added_cols[new_col_name] = consolidated_lists.apply(lambda x: x[i] if i < len(x) else None)

# Create a new DataFrame with the 'ID' and the new 'Vorname_' columns
df_pivoted_added_column = pd.DataFrame(new_added_cols)

merged_df = pd.merge(merged_df, df_pivoted_added_column, on='90999_ItemID', how='left', suffixes=('_original', '_pivoted'))

In [11]:
# Code, der Spalten einer ID ausliest, und in neue Spalten in erste Zeile umschreibt
# Geändert werden muss immer der Wert in "column_name" zur entsprechenden Spalte und Values

column_name = '5_Affiliation'

df_pivoted_add_column = merged_df.pivot_table(
    index='90999_ItemID',           # Use itemID as the index for the new DataFrame
    columns='Item_seq',    # Use vorname_seq to create new columns
    values=column_name,         # The values for the new columns 
    aggfunc='first'           # In case of multiple values for a seq (unlikely here)
)

def consolidate_values(row):
    # Get non-NaN values from the row, preserving order
    values = [value for value in row if pd.notna(value)]
    return values

# Apply the function to each row to get a list of consolidated values
consolidated_lists = df_pivoted_add_column.apply(consolidate_values, axis=1)

# Create new columns based on the consolidated lists
max_values = consolidated_lists.apply(len).max() # Find the maximum number of values in any row

new_added_cols = {}
for i in range(max_values):
    new_col_name = f'{i + 1}{column_name}'
    new_added_cols[new_col_name] = consolidated_lists.apply(lambda x: x[i] if i < len(x) else None)

# Create a new DataFrame with the 'ID' and the new 'Vorname_' columns
df_pivoted_added_column = pd.DataFrame(new_added_cols)

merged_df = pd.merge(merged_df, df_pivoted_added_column, on='90999_ItemID', how='left', suffixes=('_original', '_pivoted'))

In [12]:
# Code, der Spalten einer ID ausliest, und in neue Spalten in erste Zeile umschreibt
# Geändert werden muss immer der Wert in "column_name" zur entsprechenden Spalte und Values

column_name = '1_Quellcreator-Nachname'

df_pivoted_add_column = merged_df.pivot_table(
    index='90999_ItemID',           # Use itemID as the index for the new DataFrame
    columns='Item_seq',    # Use vorname_seq to create new columns
    values=column_name,         # The values for the new columns 
    aggfunc='first'           # In case of multiple values for a seq (unlikely here)
)

def consolidate_values(row):
    # Get non-NaN values from the row, preserving order
    values = [value for value in row if pd.notna(value)]
    return values

# Apply the function to each row to get a list of consolidated values
consolidated_lists = df_pivoted_add_column.apply(consolidate_values, axis=1)

# Create new columns based on the consolidated lists
max_values = consolidated_lists.apply(len).max() # Find the maximum number of values in any row

new_added_cols = {}
for i in range(max_values):
    new_col_name = f'{i + 1}{column_name}'
    new_added_cols[new_col_name] = consolidated_lists.apply(lambda x: x[i] if i < len(x) else None)

# Create a new DataFrame with the 'ID' and the new 'Vorname_' columns
df_pivoted_added_column = pd.DataFrame(new_added_cols)

merged_df = pd.merge(merged_df, df_pivoted_added_column, on='90999_ItemID', how='left', suffixes=('_original', '_pivoted'))

In [13]:
# Code, der Spalten einer ID ausliest, und in neue Spalten in erste Zeile umschreibt
# Geändert werden muss immer der Wert in "column_name" zur entsprechenden Spalte und Values

column_name = '2_Quellcreator-Vorname'

df_pivoted_add_column = merged_df.pivot_table(
    index='90999_ItemID',           # Use itemID as the index for the new DataFrame
    columns='Item_seq',    # Use vorname_seq to create new columns
    values=column_name,         # The values for the new columns 
    aggfunc='first'           # In case of multiple values for a seq (unlikely here)
)

def consolidate_values(row):
    # Get non-NaN values from the row, preserving order
    values = [value for value in row if pd.notna(value)]
    return values

# Apply the function to each row to get a list of consolidated values
consolidated_lists = df_pivoted_add_column.apply(consolidate_values, axis=1)

# Create new columns based on the consolidated lists
max_values = consolidated_lists.apply(len).max() # Find the maximum number of values in any row

new_added_cols = {}
for i in range(max_values):
    new_col_name = f'{i + 1}{column_name}'
    new_added_cols[new_col_name] = consolidated_lists.apply(lambda x: x[i] if i < len(x) else None)

# Create a new DataFrame with the 'ID' and the new 'Vorname_' columns
df_pivoted_added_column = pd.DataFrame(new_added_cols)

merged_df = pd.merge(merged_df, df_pivoted_added_column, on='90999_ItemID', how='left', suffixes=('_original', '_pivoted'))

In [14]:
# Code, der Spalten einer ID ausliest, und in neue Spalten in erste Zeile umschreibt
# Geändert werden muss immer der Wert in "column_name" zur entsprechenden Spalte und Values

column_name = '1_File-Description'

df_pivoted_add_column = merged_df.pivot_table(
    index='90999_ItemID',           # Use itemID as the index for the new DataFrame
    columns='Item_seq',    # Use vorname_seq to create new columns
    values=column_name,         # The values for the new columns 
    aggfunc='first'           # In case of multiple values for a seq (unlikely here)
)

def consolidate_values(row):
    # Get non-NaN values from the row, preserving order
    values = [value for value in row if pd.notna(value)]
    return values

# Apply the function to each row to get a list of consolidated values
consolidated_lists = df_pivoted_add_column.apply(consolidate_values, axis=1)

# Create new columns based on the consolidated lists
max_values = consolidated_lists.apply(len).max() # Find the maximum number of values in any row

new_added_cols = {}
for i in range(max_values):
    new_col_name = f'{i + 1}{column_name}'
    new_added_cols[new_col_name] = consolidated_lists.apply(lambda x: x[i] if i < len(x) else None)

# Create a new DataFrame with the 'ID' and the new 'Vorname_' columns
df_pivoted_added_column = pd.DataFrame(new_added_cols)

merged_df = pd.merge(merged_df, df_pivoted_added_column, on='90999_ItemID', how='left', suffixes=('_original', '_pivoted'))

In [15]:
# Code, der Spalten einer ID ausliest, und in neue Spalten in erste Zeile umschreibt
# Geändert werden muss immer der Wert in "column_name" zur entsprechenden Spalte und Values

column_name = '2_File-title'

df_pivoted_add_column = merged_df.pivot_table(
    index='90999_ItemID',           # Use itemID as the index for the new DataFrame
    columns='Item_seq',    # Use vorname_seq to create new columns
    values=column_name,         # The values for the new columns 
    aggfunc='first'           # In case of multiple values for a seq (unlikely here)
)

def consolidate_values(row):
    # Get non-NaN values from the row, preserving order
    values = [value for value in row if pd.notna(value)]
    return values

# Apply the function to each row to get a list of consolidated values
consolidated_lists = df_pivoted_add_column.apply(consolidate_values, axis=1)

# Create new columns based on the consolidated lists
max_values = consolidated_lists.apply(len).max() # Find the maximum number of values in any row

new_added_cols = {}
for i in range(max_values):
    new_col_name = f'{i + 1}{column_name}'
    new_added_cols[new_col_name] = consolidated_lists.apply(lambda x: x[i] if i < len(x) else None)

# Create a new DataFrame with the 'ID' and the new 'Vorname_' columns
df_pivoted_added_column = pd.DataFrame(new_added_cols)

merged_df = pd.merge(merged_df, df_pivoted_added_column, on='90999_ItemID', how='left', suffixes=('_original', '_pivoted'))

In [16]:
# Code, der Spalten einer ID ausliest, und in neue Spalten in erste Zeile umschreibt
# Geändert werden muss immer der Wert in "column_name" zur entsprechenden Spalte und Values

column_name = '3_File-Category'

df_pivoted_add_column = merged_df.pivot_table(
    index='90999_ItemID',           # Use itemID as the index for the new DataFrame
    columns='Item_seq',    # Use vorname_seq to create new columns
    values=column_name,         # The values for the new columns 
    aggfunc='first'           # In case of multiple values for a seq (unlikely here)
)

def consolidate_values(row):
    # Get non-NaN values from the row, preserving order
    values = [value for value in row if pd.notna(value)]
    return values

# Apply the function to each row to get a list of consolidated values
consolidated_lists = df_pivoted_add_column.apply(consolidate_values, axis=1)

# Create new columns based on the consolidated lists
max_values = consolidated_lists.apply(len).max() # Find the maximum number of values in any row

new_added_cols = {}
for i in range(max_values):
    new_col_name = f'{i + 1}{column_name}'
    new_added_cols[new_col_name] = consolidated_lists.apply(lambda x: x[i] if i < len(x) else None)

# Create a new DataFrame with the 'ID' and the new 'Vorname_' columns
df_pivoted_added_column = pd.DataFrame(new_added_cols)

merged_df = pd.merge(merged_df, df_pivoted_added_column, on='90999_ItemID', how='left', suffixes=('_original', '_pivoted'))

In [17]:
# Code, der Spalten einer ID ausliest, und in neue Spalten in erste Zeile umschreibt
# Geändert werden muss immer der Wert in "column_name" zur entsprechenden Spalte und Values

column_name = '4_File-content'

df_pivoted_add_column = merged_df.pivot_table(
    index='90999_ItemID',           # Use itemID as the index for the new DataFrame
    columns='Item_seq',    # Use vorname_seq to create new columns
    values=column_name,         # The values for the new columns 
    aggfunc='first'           # In case of multiple values for a seq (unlikely here)
)

def consolidate_values(row):
    # Get non-NaN values from the row, preserving order
    values = [value for value in row if pd.notna(value)]
    return values

# Apply the function to each row to get a list of consolidated values
consolidated_lists = df_pivoted_add_column.apply(consolidate_values, axis=1)

# Create new columns based on the consolidated lists
max_values = consolidated_lists.apply(len).max() # Find the maximum number of values in any row

new_added_cols = {}
for i in range(max_values):
    new_col_name = f'{i + 1}{column_name}'
    new_added_cols[new_col_name] = consolidated_lists.apply(lambda x: x[i] if i < len(x) else None)

# Create a new DataFrame with the 'ID' and the new 'Vorname_' columns
df_pivoted_added_column = pd.DataFrame(new_added_cols)

merged_df = pd.merge(merged_df, df_pivoted_added_column, on='90999_ItemID', how='left', suffixes=('_original', '_pivoted'))

In [18]:
# Code, der Spalten einer ID ausliest, und in neue Spalten in erste Zeile umschreibt
# Geändert werden muss immer der Wert in "column_name" zur entsprechenden Spalte und Values

column_name = '5_File-visibility'

df_pivoted_add_column = merged_df.pivot_table(
    index='90999_ItemID',           # Use itemID as the index for the new DataFrame
    columns='Item_seq',    # Use vorname_seq to create new columns
    values=column_name,         # The values for the new columns 
    aggfunc='first'           # In case of multiple values for a seq (unlikely here)
)

def consolidate_values(row):
    # Get non-NaN values from the row, preserving order
    values = [value for value in row if pd.notna(value)]
    return values

# Apply the function to each row to get a list of consolidated values
consolidated_lists = df_pivoted_add_column.apply(consolidate_values, axis=1)

# Create new columns based on the consolidated lists
max_values = consolidated_lists.apply(len).max() # Find the maximum number of values in any row

new_added_cols = {}
for i in range(max_values):
    new_col_name = f'{i + 1}{column_name}'
    new_added_cols[new_col_name] = consolidated_lists.apply(lambda x: x[i] if i < len(x) else None)

# Create a new DataFrame with the 'ID' and the new 'Vorname_' columns
df_pivoted_added_column = pd.DataFrame(new_added_cols)

merged_df = pd.merge(merged_df, df_pivoted_added_column, on='90999_ItemID', how='left', suffixes=('_original', '_pivoted'))

In [19]:
# Code, der Spalten einer ID ausliest, und in neue Spalten in erste Zeile umschreibt
# Geändert werden muss immer der Wert in "column_name" zur entsprechenden Spalte und Values

column_name = '6_File-OA'

df_pivoted_add_column = merged_df.pivot_table(
    index='90999_ItemID',           # Use itemID as the index for the new DataFrame
    columns='Item_seq',    # Use vorname_seq to create new columns
    values=column_name,         # The values for the new columns 
    aggfunc='first'           # In case of multiple values for a seq (unlikely here)
)

def consolidate_values(row):
    # Get non-NaN values from the row, preserving order
    values = [value for value in row if pd.notna(value)]
    return values

# Apply the function to each row to get a list of consolidated values
consolidated_lists = df_pivoted_add_column.apply(consolidate_values, axis=1)

# Create new columns based on the consolidated lists
max_values = consolidated_lists.apply(len).max() # Find the maximum number of values in any row

new_added_cols = {}
for i in range(max_values):
    new_col_name = f'{i + 1}{column_name}'
    new_added_cols[new_col_name] = consolidated_lists.apply(lambda x: x[i] if i < len(x) else None)

# Create a new DataFrame with the 'ID' and the new 'Vorname_' columns
df_pivoted_added_column = pd.DataFrame(new_added_cols)

merged_df = pd.merge(merged_df, df_pivoted_added_column, on='90999_ItemID', how='left', suffixes=('_original', '_pivoted'))

In [20]:
# Code, der Spalten einer ID ausliest, und in neue Spalten in erste Zeile umschreibt
# Geändert werden muss immer der Wert in "column_name" zur entsprechenden Spalte und Values

column_name = '7_File-Storage'

df_pivoted_add_column = merged_df.pivot_table(
    index='90999_ItemID',           # Use itemID as the index for the new DataFrame
    columns='Item_seq',    # Use vorname_seq to create new columns
    values=column_name,         # The values for the new columns 
    aggfunc='first'           # In case of multiple values for a seq (unlikely here)
)

def consolidate_values(row):
    # Get non-NaN values from the row, preserving order
    values = [value for value in row if pd.notna(value)]
    return values

# Apply the function to each row to get a list of consolidated values
consolidated_lists = df_pivoted_add_column.apply(consolidate_values, axis=1)

# Create new columns based on the consolidated lists
max_values = consolidated_lists.apply(len).max() # Find the maximum number of values in any row

new_added_cols = {}
for i in range(max_values):
    new_col_name = f'{i + 1}{column_name}'
    new_added_cols[new_col_name] = consolidated_lists.apply(lambda x: x[i] if i < len(x) else None)

# Create a new DataFrame with the 'ID' and the new 'Vorname_' columns
df_pivoted_added_column = pd.DataFrame(new_added_cols)

merged_df = pd.merge(merged_df, df_pivoted_added_column, on='90999_ItemID', how='left', suffixes=('_original', '_pivoted'))

In [21]:
# Code, der Spalten einer ID ausliest, und in neue Spalten in erste Zeile umschreibt
# Geändert werden muss immer der Wert in "column_name" zur entsprechenden Spalte und Values

column_name = '8_File-Licence'

df_pivoted_add_column = merged_df.pivot_table(
    index='90999_ItemID',           # Use itemID as the index for the new DataFrame
    columns='Item_seq',    # Use vorname_seq to create new columns
    values=column_name,         # The values for the new columns 
    aggfunc='first'           # In case of multiple values for a seq (unlikely here)
)

def consolidate_values(row):
    # Get non-NaN values from the row, preserving order
    values = [value for value in row if pd.notna(value)]
    return values

# Apply the function to each row to get a list of consolidated values
consolidated_lists = df_pivoted_add_column.apply(consolidate_values, axis=1)

# Create new columns based on the consolidated lists
max_values = consolidated_lists.apply(len).max() # Find the maximum number of values in any row

new_added_cols = {}
for i in range(max_values):
    new_col_name = f'{i + 1}{column_name}'
    new_added_cols[new_col_name] = consolidated_lists.apply(lambda x: x[i] if i < len(x) else None)

# Create a new DataFrame with the 'ID' and the new 'Vorname_' columns
df_pivoted_added_column = pd.DataFrame(new_added_cols)

merged_df = pd.merge(merged_df, df_pivoted_added_column, on='90999_ItemID', how='left', suffixes=('_original', '_pivoted'))

In [22]:
# Code, der Spalten einer ID ausliest, und in neue Spalten in erste Zeile umschreibt
# Geändert werden muss immer der Wert in "column_name" zur entsprechenden Spalte und Values

column_name = '9_File-Embargo'

df_pivoted_add_column = merged_df.pivot_table(
    index='90999_ItemID',           # Use itemID as the index for the new DataFrame
    columns='Item_seq',    # Use vorname_seq to create new columns
    values=column_name,         # The values for the new columns 
    aggfunc='first'           # In case of multiple values for a seq (unlikely here)
)

def consolidate_values(row):
    # Get non-NaN values from the row, preserving order
    values = [value for value in row if pd.notna(value)]
    return values

# Apply the function to each row to get a list of consolidated values
consolidated_lists = df_pivoted_add_column.apply(consolidate_values, axis=1)

# Create new columns based on the consolidated lists
max_values = consolidated_lists.apply(len).max() # Find the maximum number of values in any row

new_added_cols = {}
for i in range(max_values):
    new_col_name = f'{i + 1}{column_name}'
    new_added_cols[new_col_name] = consolidated_lists.apply(lambda x: x[i] if i < len(x) else None)

# Create a new DataFrame with the 'ID' and the new 'Vorname_' columns
df_pivoted_added_column = pd.DataFrame(new_added_cols)

merged_df = pd.merge(merged_df, df_pivoted_added_column, on='90999_ItemID', how='left', suffixes=('_original', '_pivoted'))

# Vor dem Umschreiben der Local Tags in eigene Spalten werden bestimmte Werte entfernt, die für die Statistiken reine Relevanz haben

## hier sind lokale Anpassungen nötig

In [23]:
for col in merged_df.columns:
    # Check if the column is of object or string dtype
    if merged_df[col].dtype == 'object':
        # Create a boolean mask for cells containing the specific text
        mask = merged_df[col].astype(str).str.contains('batch add local tags', na=False)
        # Replace the values in the masked cells with an empty string or None
        merged_df.loc[mask, col] = '' # Or use pd.NA or None if you prefer missing values

In [24]:
for col in merged_df.columns:
    # Check if the column is of object or string dtype
    if merged_df[col].dtype == 'object':
        # Create a boolean mask for cells containing the specific text
        mask = merged_df[col].astype(str).str.contains('batch change', na=False)
        # Replace the values in the masked cells with an empty string or None
        merged_df.loc[mask, col] = '' # Or use pd.NA or None if you prefer missing values

In [25]:
for col in merged_df.columns:
    # Check if the column is of object or string dtype
    if merged_df[col].dtype == 'object':
        # Create a boolean mask for cells containing the specific text
        mask = merged_df[col].astype(str).str.contains('multiple_import', na=False)
        # Replace the values in the masked cells with an empty string or None
        merged_df.loc[mask, col] = '' # Or use pd.NA or None if you prefer missing values

In [27]:
for col in merged_df.columns:
    # Check if the column is of object or string dtype
    if merged_df[col].dtype == 'object':
        # Create a boolean mask for cells containing the specific text
        mask = merged_df[col].astype(str).str.contains('Yearbook', na=False)
        # Replace the values in the masked cells with an empty string or None
        merged_df.loc[mask, col] = '' # Or use pd.NA or None if you prefer missing values

In [28]:
for col in merged_df.columns:
    # Check if the column is of object or string dtype
    if merged_df[col].dtype == 'object':
        # Create a boolean mask for cells containing the specific text
        mask = merged_df[col].astype(str).str.contains('eDoc Migration', na=False)
        # Replace the values in the masked cells with an empty string or None
        merged_df.loc[mask, col] = '' # Or use pd.NA or None if you prefer missing values

In [29]:
for col in merged_df.columns:
    # Check if the column is of object or string dtype
    if merged_df[col].dtype == 'object':
        # Create a boolean mask for cells containing the specific text
        mask = merged_df[col].astype(str).str.contains('Daten2011', na=False)
        # Replace the values in the masked cells with an empty string or None
        merged_df.loc[mask, col] = '' # Or use pd.NA or None if you prefer missing values

In [30]:
for col in merged_df.columns:
    # Check if the column is of object or string dtype
    if merged_df[col].dtype == 'object':
        # Create a boolean mask for cells containing the specific text
        mask = merged_df[col].astype(str).str.contains('MPIfG_new', na=False)
        # Replace the values in the masked cells with an empty string or None
        merged_df.loc[mask, col] = '' # Or use pd.NA or None if you prefer missing values

In [31]:
# Übersicht der noch vorhandenen Local Tags, um zu entscheiden, ob weitere Bereinigungen notwendig sind
print(merged_df['0_LocalTags'].unique())

['Hurz' 'MPIfG_journal-article' nan 'MPIfG_climate' '' 'MPIfG_IMPRS'
 'MPIfG_book' 'OA_Bronze' 'MPIfG_WS_WissPol' 'MPIfG_WS_UeArt'
 'MPIfG_WS_ProRe' 'MPIfG_WS_Deb' 'OA_Diamond' 'MPIfG_WS_Theo' 'MPIfG_SUV'
 'MPIfG_dp' 'OA_S2O' 'journal article' 'MaxPo_journal-article' 'MPIfG_UEF']


In [33]:
# Jetzt wird für jeden Local Tag ein eigene Spalte erstellt 

# Identify valid tags based on a condition (e.g., if they are strings and not purely numeric)
def is_word_tag(value):
    if isinstance(value, str):
        return any(c.isalpha() for c in value)
    return False

# Create the list of valid word-based tags from the 'label' column
valid_tags = merged_df['0_LocalTags'].dropna().apply(lambda x: x if is_word_tag(x) else None).dropna().unique().tolist()

print(f"Identified valid tags: {valid_tags}")

# Initialize an empty list to store the dictionaries for the new DataFrame
all_new_records = []

# Group the DataFrame by 'ID'
grouped = merged_df.groupby('90999_ItemID')

# Iterate through each group
for id_value, group_df in grouped:
    # Filter the group to include only rows with valid word tags in the 'label' column
    group_df_filtered = group_df[group_df['0_LocalTags'].isin(valid_tags)].copy() # Use .copy()

    # Sort the filtered group alphabetically by the 'label' column
    group_df_sorted = group_df_filtered.sort_values(by='0_LocalTags')

    # Skip processing if there are no valid tags for this ID after filtering
    if group_df_sorted.empty:
        continue

    new_columns_data = {'90999_ItemID': id_value} # Include the ID in the new record
    tag_counts = defaultdict(int) # To track how many times a tag appears for this ID

    # Iterate through the sorted and filtered rows in the group
    for index, row in group_df_sorted.iterrows():
        label_value = row['0_LocalTags']
        # We will use the label_value as the content of the new column
        # The sequenced_value is not used as the column content in this version

        tag_counts[label_value] += 1 # Increment the count for this tag

        # Create the new column name (header)
        # Use the tag name. If it appears more than once, add a counter suffix.
        if tag_counts[label_value] > 1:
            new_column_name = f'0550_{[label_value]}'
        else:
            new_column_name = f'0550_{[label_value]}'

        # Add the label_value as the content for the new column header
        new_columns_data[new_column_name] = label_value # Put the label_value in the column

    # Add the dictionary for this ID to the list
    all_new_records.append(new_columns_data)

# Create the final DataFrame
df_tags = pd.DataFrame(all_new_records)

Identified valid tags: ['Hurz', 'MPIfG_journal-article', 'MPIfG_climate', 'MPIfG_IMPRS', 'MPIfG_book', 'OA_Bronze', 'MPIfG_WS_WissPol', 'MPIfG_WS_UeArt', 'MPIfG_WS_ProRe', 'MPIfG_WS_Deb', 'OA_Diamond', 'MPIfG_WS_Theo', 'MPIfG_SUV', 'MPIfG_dp', 'OA_S2O', 'journal article', 'MaxPo_journal-article', 'MPIfG_UEF']


In [34]:
# Hinzufügen der Tag spalten zum bestehenden Dataframe

merged_df = pd.merge(merged_df, df_tags, on='90999_ItemID', how='left', suffixes=('_original', '_pivoted'))

In [35]:
# Zeilen mit den umgeschriebenen Werten alle löschen (für jede ID nur noch eine Zeile)

df = merged_df.drop_duplicates(subset=['90999_ItemID'], keep='first')

In [36]:
# Auswahl der exportierenden Spalten

specific_columns1 = ['1002_Genre', '1001_Titel', '0013_Creator-Rolle']
nachname_columns = [col for col in df.columns if '1_Nachname' in col]
vorname_columns = [col for col in df.columns if '2_Vorname' in col]
coneID_columns = [col for col in df.columns if '4_Cone_ID' in col]
affiliation_columns = [col for col in df.columns if '5_Affiliation' in col]
status_columns = [col for col in df.columns if '6_Status' in col]
specific_columns2 = ['1101_Quellentitel', '1100_Quellen-Genre','1110_Quellcreator-Rolle']
file_creator_columns = [col for col in df.columns if 'Quellcreator-Nachname' in col]
file_creator_columns2 = [col for col in df.columns if 'Quellcreator-Vorname' in col]
specific_columns3 = [  '1104_Band','1105_Heft','1106_Anfangsseite','1107_Endseite', '1108_Sequenznummer',
                       '1103_Ersterscheinungsdatum',
                       '2201_Identifier','2202_ID','1109_Identifier',
                       '2200_Sprache',
                       '1108_Review', '1021_Degree']
file_columns =  [col for col in df.columns if '_File' in col]
local_tags_columns =  [col for col in df.columns if '0550' in col]
info_columns = [col for col in df.columns if '9099' in col]

all_desired_columns = specific_columns1 + nachname_columns + vorname_columns + coneID_columns + affiliation_columns + status_columns + specific_columns2 + file_creator_columns + file_creator_columns2 + specific_columns3 + file_columns + local_tags_columns + info_columns

df_cropped = df[all_desired_columns]

In [37]:
df_cropped.columns

Index(['1002_Genre', '1001_Titel', '0013_Creator-Rolle', '1_Nachname',
       '11_Nachname', '21_Nachname', '31_Nachname', '41_Nachname',
       '51_Nachname', '61_Nachname',
       ...
       '0550_['MPIfG_climate']', '0550_['MPIfG_WS_Deb']',
       '0550_['MaxPo_journal-article']', '0550_['OA_Bronze']',
       '0550_['OA_S2O']', '0550_['MPIfG_WS_UeArt']',
       '0550_['MPIfG_WS_WissPol']', '90999_ItemID', '90992_Stichworte',
       '90991_Abstract'],
      dtype='object', length=226)

In [38]:
# Zur Sortierung der Spalten werden jetzt die Header mit Zahlen angereichert

# Add a prefix "40004" to all column names that include the word "File"
df_cropped.rename(
    columns=lambda col: f"400004{col}" if "File" in col else col,
    inplace=True
)

# Add a prefix "101" to all column names that include the word "Quellcreator"
df_cropped.rename(
    columns=lambda col: f"101{col}" if "Quellcreator-Nachname" in col else col,
    inplace=True
)

# Add a prefix "101" to all column names that include the word "Quellcreator"
df_cropped.rename(
    columns=lambda col: f"101{col}" if "Quellcreator-Vorname" in col else col,
    inplace=True
)


# Add a prefix "50" to all column names that include the string "0550_"
df_cropped.rename(
    columns=lambda col: f"50{col}" if "0550_" in col else col,
    inplace=True
)



C:\Users\cgmol\AppData\Local\Temp\ipykernel_72184\3864308127.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cropped.rename(
C:\Users\cgmol\AppData\Local\Temp\ipykernel_72184\3864308127.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cropped.rename(
C:\Users\cgmol\AppData\Local\Temp\ipykernel_72184\3864308127.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cropped.rename(
C:\Users\cgmol\AppData\Local\Temp\ipykern

In [39]:
# durch Umschreiben redundant gewordene Spalten löschen

df_cropped.drop(columns=['1_Nachname', '2_Vorname', '5_Affiliation'], inplace=True)
df_cropped.drop(columns=["1011_Quellcreator-Nachname", "1012_Quellcreator-Vorname"], inplace=True)

C:\Users\cgmol\AppData\Local\Temp\ipykernel_72184\1670141528.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cropped.drop(columns=['1_Nachname', '2_Vorname', '5_Affiliation'], inplace=True)
C:\Users\cgmol\AppData\Local\Temp\ipykernel_72184\1670141528.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cropped.drop(columns=["1011_Quellcreator-Nachname", "1012_Quellcreator-Vorname"], inplace=True)


In [43]:
df_cropped.columns = df_cropped.columns.astype(str)

def extract_numeric(col_name):
    match = re.search(r'(\d+)', str(col_name))
    if match:
        return (int(match.group(1)), col_name)  # (Zahl, Originalname) für stabile Sortierung
    else:
        return (float('inf'), col_name)
    
sorted_columns = sorted(df_cropped.columns, key=extract_numeric)

# Reorder the DataFrame
df_sorted = df_cropped[sorted_columns]

# Display the sorted DataFrame
df_sorted

,11_Nachname,12_Vorname,0013_Creator-Rolle,15_Affiliation,21_Nachname,22_Vorname,25_Affiliation,31_Nachname,32_Vorname,35_Affiliation,...,400004104_File-content,400004105_File-visibility,400004106_File-OA,400004107_File-Storage,400004111_File-Description,400004112_File-title,400004113_File-Category,400004114_File-content,400004115_File-visibility,400004117_File-Storage
0,Cigna,Luca,AUTHOR,"Politische Ökonomie, MPI for the Study of Soci...",Lopez-Uroz,Nina,Umstrittene Ökologien in kapitalistischen Demo...,None,None,None,...,None,None,None,None,None,None,None,None,None,None
6,Gruber,Stephan,AUTHOR,"Wirtschaftssoziologie, MPI for the Study of So...",None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
9,Schwan,Michael,AUTHOR,Department of Global Economics and Management ...,Trampusch,Christine,"Cologne Center for Comparative Politics, Unive...",Horn,Jonas L.,International Max Planck Research School on th...,...,None,None,None,None,None,None,None,None,None,None
15,Polyák,Pálma,AUTHOR,"Politische Ökonomie, MPI for the Study of Soci...",None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
18,Kullick,Paul Niklas,AUTHOR,International Max Planck Research School on th...,Petry,Johannes,"Institute for Political Science, Goethe Univer...",None,None,"Institute for Political Science, Goethe Univer...",...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2314,Marktanner,Alina,AUTHOR,International Max Planck Research School on th...,None,None,"RWTH Aachen, Germany",None,None,None,...,None,None,None,None,None,None,None,None,None,None
2318,Neimanns,Erik,AUTHOR,"Politische Ökonomie von Wachstumsmodellen, MPI...",None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2323,Wansleben,Leon,AUTHOR,"Soziologie öffentlicher Finanzen und Schulden,...",None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2327,Braun,Benjamin,AUTHOR,"Wirtschaftssoziologie, MPI for the Study of So...",Koddenbrock,Kai,"Bayreuth University, Germany",None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [44]:
# Kopie der Basisrohdaten für Excel
df_sorted.to_excel('Daten/Basisrohdaten_PuRe.xlsx', index=False)